In [1]:
import cv2

In [1]:
import cv2

# Initialize video capture (use 0 for default webcam)
cap = cv2.VideoCapture(0)

# Check if the video capture is opened correctly
if not cap.isOpened():
    print("Error: Could not open video capture device.")
    exit()

def Preprocessing(img):
    if img is None:
        print("Error: Received an empty image.")
        return None
    
    imgPre = cv2.GaussianBlur(img, (5, 5), 3)
    return imgPre

while True:
    success, img = cap.read()
    if not success:
        print("Error: Failed to read frame from video capture.")
        break

    imgPre = Preprocessing(img)
    if imgPre is not None:
        cv2.imshow("ImagePre", imgPre)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import cvzone
import numpy as np
from cvzone.ColorModule import ColorFinder
cap=cv2.VideoCapture(1)
cap.set(3,640)
cap.set(4,480)
totalMoney = 0



myColorFinder = ColorFinder(False)
# custom Orange color
hsvVals={'hmin':0, 'smin':0, 'vmin':145, 'hmax' :63 , 'smax':91, 'vmax':255}





def empty(a) :
    pass

cv2.namedWindow("Settings")
cv2.resizeWindow("Settings",640,240)
cv2.createTrackbar("Threshold1","Settings",50,255,empty)
cv2.createTrackbar("Threshold2","Settings",100,255,empty)

def Preprocessing(img):
    imgPre = cv2.GaussianBlur(img,(5,5),3)
    thresh1=cv2.getTrackbarPos("Threshold1","Settings")
    thresh2=cv2.getTrackbarPos("Threshold2","Settings")
    imgPre = cv2.Canny(imgPre,thresh1,thresh2)
    kernel = np.ones((3,3),np.uint8)
    imgPre =cv2.dilate(imgPre.kernel,iterations=1)
    imgPre =cv2.morphologyEx(imgPre, cv2.MOPH_CLOSE, kernel)
    return imgPre

while True:
    sucess,img=cap.read()
    imgPre= Preprocessing(img)
    imgContours,conFound=cvzone.findContours(img,imgPre,minArea=20)
    imgContours, conFound = cvzone.findContours(img,imgPre,minArea=20)

    totalMoney =0
    imgCount =np.zeros((480,640,3),np.uint8)
    if conFound:
        for count,contour in enumerate(conFound):
            peri =cv2.arcLength(contour['cnt'], True)
            approx = cv2.approxpolyDp(contour['cnt'], 0.02 * peri, True)
          
            if len(approx)>5:
               area = contour['area']>5
               x,y,w,h=contour['bbox']
               imgCrop = img[y:y+h,x:x+w]
               # cv2.imshow(str(count),imgCrop)
    
               imgColor,mask = myColorFinder.update(imgCrop,hsvVals) 
               whitePixelCount = cv2.countNonZero(mask)
               print(whitePixelCount)
               if area<2050:
                   totalMoney +=5
               elif 2050<area<2500:
                    totalMoney +=1
               else:
                     totalMoney +=2
    # print(totalMoney)
    cvzone.putTextRect(imgCount,f'Rs.{totalMoney}' , (100,200),scale=10,affsat=30,thickness=7)
    imgStacked= cvzone.stackImages([img,imgPre,imgContours,imgCount],2,1)

    cvzone.putTextRect(imgCount,f' Rs.{totalMoney}',(50,50),scale=10)
    success, img =cap.read()
    cv2.imshow("Image",  imgStacked) 
    cv2.imshow("imgColor", imgColor)
    cv2.waitKey(1)
